In [ ]:
from pathlib import Path
import pandas as pd
from datetime import datetime

# =========================
# PROJECT PATHS
# =========================

PROJECT_ROOT = Path.cwd().parents[1]
DATA_FEB_DIR = PROJECT_ROOT / "Dataset_feb"
SAMSUNG_DIR = DATA_FEB_DIR / "Dataset_samsung"

print("Samsung data folder:", SAMSUNG_DIR)
print("Folder found:", SAMSUNG_DIR.exists())


# =========================
# EMAIL-TO-ID MAPPING
# =========================

email_to_id = {
    "info+elbn_pt_1@gamebus.eu": 1,
    "info+elbn_pt_6@gamebus.eu": 2,
    "info+elbn_pt_35@gamebus.eu": 3,
    "info+elbn_pt_24@gamebus.eu": 4,
    "info+elbn_pt_39@gamebus.eu": 5,
    "info+elbn_pt_12@gamebus.eu": 6,
    "info+elbn_pt_10@gamebus.eu": 7,
    "info+elbn_pt_18@gamebus.eu": 8,
    "info+elbn_pt_21@gamebus.eu": 9,
    "info+elbn_pt_9@gamebus.eu": 10,
    "info+elbn_pt_32@gamebus.eu": 11,
    "info+elbn_pt_40@gamebus.eu": 12,
    "info+elbn_pt_11@gamebus.eu": 13,
    "info+elbn_pt_4@gamebus.eu": 14,
    "info+elbn_pt_19@gamebus.eu": 15,
    "info+elbn_pt_5@gamebus.eu": 16,
    "info+elbn_pt_16@gamebus.eu": 17,
    "info+elbn_pt_8@gamebus.eu": 18,
    "info+elbn_pt_15@gamebus.eu": 19,
    "info+elbn_pt_31@gamebus.eu": 20,
    "info+elbn_pt_25@gamebus.eu": 21,
    "info+elbn_pt_26@gamebus.eu": 22,
    "info+elbn_pt_13@gamebus.eu": 23,
    "info+elbn_pt_27@gamebus.eu": 24,
    "info+elbn_pt_33@gamebus.eu": 25,
    "info+elbn_pt_2@gamebus.eu": 26,
    "info+elbn_pt_28@gamebus.eu": 27,
    "info+elbn_pt_37@gamebus.eu": 28,
    "info+elbn_pt_38@gamebus.eu": 29,
    "info+elbn_pt_34@gamebus.eu": 30,
    "info+elbn_pt_30@gamebus.eu": 31,
    "info+elbn_pt_17@gamebus.eu": 32,
    "info+elbn_pt_29@gamebus.eu": 33,
    "info+elbn_pt_22@gamebus.eu": 34,
    "info+elbn_pt_36@gamebus.eu": 35,
    "info+elbn_pt_3@gamebus.eu": 36,
    "info+elbn_pt_14@gamebus.eu": 37
}

print("Mapping loaded successfully!")

Data Cleaning: Activity, Mood, Social

In [ ]:
# =========================
# ACTIVITY FILE
# =========================

activity_file_path = SAMSUNG_DIR / "query_result_2025-04-01T12_17_44.808327034+02_00_Tizen.csv"
activity_file = pd.read_csv(activity_file_path)

activity_file["Event Created At"] = pd.to_datetime(activity_file["Event Created At"])
filtered_activity_file = activity_file.copy()

filtered_activity_file = filtered_activity_file.reset_index(drop=True)

for i in range(len(filtered_activity_file)):
    filtered_activity_file.loc[i, "User ID"] = email_to_id.get(
        filtered_activity_file.loc[i, "Email"]
    )

filtered_activity_file = filtered_activity_file[
    ["User ID", "Event Created At", "Property", "Property Instance"]
]

filtered_activity_file = filtered_activity_file.rename(
    columns={
        "User ID": "user_id",
        "Event Created At": "timestamp",
        "Property Instance": "property_instance"
    }
)


# =========================
# MOOD FILE
# =========================

mood_file_path = SAMSUNG_DIR / "query_result_2025-04-01T12_39_52.079680538+02_00_LOG_MOOD.csv"
mood_file = pd.read_csv(mood_file_path)

filtered_rows = []

for line in range(4, len(mood_file), 5):
    ts = mood_file.iloc[line]["Property Instance"]
    ts = datetime.fromtimestamp(ts / 1000).strftime("%Y-%m-%dT%H:%M:%S")

    row = {
        "user_id": email_to_id.get(mood_file.iloc[line]["Email"]),
        "timestamp": ts,
        "valence_value": mood_file.iloc[line - 4]["Property Instance"],
        "arousal_value": mood_file.iloc[line - 3]["Property Instance"],
        "stress_value": mood_file.iloc[line - 2]["Property Instance"],
        "convenience": mood_file.iloc[line - 1]["Property Instance"]
    }

    filtered_rows.append(row)

filtered_mood_file = pd.DataFrame(filtered_rows)
filtered_mood_file = filtered_mood_file.sort_values(
    by=["user_id", "timestamp"]
)

cols_to_check = [
    "valence_value",
    "arousal_value",
    "stress_value",
    "convenience"
]

filtered_mood_file = filtered_mood_file[
    ~(filtered_mood_file[cols_to_check] == -1).any(axis=1)
]


# =========================
# SOCIAL FILE
# =========================

social_file_path = SAMSUNG_DIR / "query_result_2025-04-01T12_40_54.021782533+02_00_social_context.csv"
social_file = pd.read_csv(social_file_path)

filtered_social_file = social_file.copy()
filtered_social_file = filtered_social_file.reset_index(drop=True)

for i in range(len(filtered_social_file)):
    filtered_social_file.loc[i, "User ID"] = email_to_id.get(
        filtered_social_file.loc[i, "Email"]
    )

filtered_social_file = filtered_social_file[
    ["User ID", "Event Created At", "Property Instance"]
]

filtered_social_file = filtered_social_file.rename(
    columns={
        "User ID": "user_id",
        "Event Created At": "timestamp",
        "Property Instance": "social_context"
    }
)

filtered_social_file = filtered_social_file.sort_values(
    by=["user_id", "timestamp"]
)

filtered_social_file = filtered_social_file.drop_duplicates(
    subset=["user_id", "timestamp", "social_context"]
)

filtered_social_file = filtered_social_file.reset_index(drop=True)


# =========================
# SAVE PROCESSED FILES
# =========================

filtered_activity_file.to_csv(
    SAMSUNG_DIR / "Initial_activity_file_New.csv",
    index=False
)

filtered_mood_file.to_csv(
    SAMSUNG_DIR / "Mood_file_New.csv",
    index=False
)

filtered_social_file.to_csv(
    SAMSUNG_DIR / "Social_file_New.csv",
    index=False
)

print("Data filtered successfully: Activity, Mood and Social files created!")

Start and End dates for each participant

In [ ]:
target_files = [
    "Initial_activity_file_New.csv",
    "Mood_file_New.csv",
    "Social_file_New.csv"
]

for current_file in target_files:
    file_path = SAMSUNG_DIR / current_file
    
    if file_path.exists():
        print("======================================================")
        print(f" TEMPORAL PROFILE SUMMARY: {current_file}")
        print("======================================================")
        
        # Load the dataset
        df = pd.read_csv(file_path)
        
        # Convert timestamps to datetime
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        
        # Get the first and last recording timestamp for each participant
        temporal_summary = (
            df.groupby("user_id")["timestamp"]
            .agg(
                Start_Timestamp="min",
                End_Timestamp="max"
            )
            .reset_index()
        )
        
        # Display the summary
        print(temporal_summary.to_string(index=False))
        print("\n")
        
    else:
        print(f"[ERROR] File not found: {file_path}\n")

Maximum time interval of continuous data acquisition

In [ ]:
# Define the target files for the streak analysis
target_files = [
    "Initial_activity_file_New.csv",
    "Mood_file_New.csv",
    "Social_file_New.csv"
]

for current_file in target_files:
    file_path = SAMSUNG_DIR / current_file
    
    if file_path.exists():
        print("====================================================================")
        print(f" MAXIMUM CONSECUTIVE DAYS (STREAK) WITH DATES: {current_file}")
        print("====================================================================")
        
        # Load the dataset
        df = pd.read_csv(file_path)
        
        # Convert timestamps to dates
        df["date"] = pd.to_datetime(df["timestamp"]).dt.normalize()
        
        # Keep only unique recording days for each participant
        unique_days = (
            df[["user_id", "date"]]
            .drop_duplicates()
            .sort_values(["user_id", "date"])
        )
        
        # Calculate the difference in days between consecutive records
        unique_days["day_diff"] = (
            unique_days.groupby("user_id")["date"]
            .diff()
            .dt.days
        )
        
        # Identify streaks:
        # a new streak starts whenever the difference is not exactly one day
        unique_days["streak_id"] = (
            unique_days["day_diff"] != 1
        ).cumsum()
        
        # Calculate streak length, start date and end date
        streak_stats = (
            unique_days.groupby(["user_id", "streak_id"])
            .agg(
                consecutive_days=("date", "size"),
                start_date=("date", "min"),
                end_date=("date", "max")
            )
            .reset_index()
        )
        
        # Find the longest streak for each participant
        max_streak_idx = (
            streak_stats.groupby("user_id")["consecutive_days"]
            .idxmax()
        )
        
        max_streaks = (
            streak_stats.loc[max_streak_idx]
            .drop(columns=["streak_id"])
        )
        
        # Rename columns for readability
        max_streaks.rename(
            columns={
                "consecutive_days": "Max_Streak_Days",
                "start_date": "Streak_Start_Date",
                "end_date": "Streak_End_Date"
            },
            inplace=True
        )
        
        # Format dates as YYYY-MM-DD
        max_streaks["Streak_Start_Date"] = (
            max_streaks["Streak_Start_Date"]
            .dt.strftime("%Y-%m-%d")
        )
        
        max_streaks["Streak_End_Date"] = (
            max_streaks["Streak_End_Date"]
            .dt.strftime("%Y-%m-%d")
        )
        
        # Display the summary
        print(max_streaks.to_string(index=False))
        print("\n")
        
    else:
        print(f"[ERROR] File not found: {file_path}\n")

Maximum time interval of continuous data acquisition for the 3 files

In [ ]:
target_files = [
    "Initial_activity_file_New.csv",
    "Mood_file_New.csv",
    "Social_file_New.csv"
]

dfs_days = []
all_files_exist = True

# Load the recording days available in each file
for file_name in target_files:
    file_path = SAMSUNG_DIR / file_name

    if file_path.exists():
        df = pd.read_csv(file_path)

        # Extract the date from each timestamp
        df["date"] = pd.to_datetime(df["timestamp"]).dt.normalize()

        # Keep only unique recording days for each participant
        unique_days = df[["user_id", "date"]].drop_duplicates()
        dfs_days.append(unique_days)

    else:
        print(f"[ERROR] File not found: {file_path}")
        all_files_exist = False


if all_files_exist and len(dfs_days) == 3:

    # Keep only the days that are present in all three datasets
    intersected_days = dfs_days[0]

    for i in range(1, len(dfs_days)):
        intersected_days = pd.merge(
            intersected_days,
            dfs_days[i],
            on=["user_id", "date"],
            how="inner"
        )

    # Sort by participant and date
    intersected_days = intersected_days.sort_values(
        ["user_id", "date"]
    )

    if intersected_days.empty:
        print(
            "There are no days in which participants have data "
            "in all three files simultaneously."
        )

    else:
        # Calculate the difference in days between consecutive records
        intersected_days["day_diff"] = (
            intersected_days.groupby("user_id")["date"]
            .diff()
            .dt.days
        )

        # Identify streaks:
        # a new streak starts whenever the difference is not exactly one day
        intersected_days["streak_id"] = (
            intersected_days["day_diff"] != 1
        ).cumsum()

        # Calculate streak length, start date and end date
        streak_stats = (
            intersected_days.groupby(["user_id", "streak_id"])
            .agg(
                consecutive_days=("date", "size"),
                start_date=("date", "min"),
                end_date=("date", "max")
            )
            .reset_index()
        )

        # Find the longest simultaneous streak for each participant
        max_streak_idx = (
            streak_stats.groupby("user_id")["consecutive_days"]
            .idxmax()
        )

        max_streaks = (
            streak_stats.loc[max_streak_idx]
            .drop(columns=["streak_id"])
        )

        # Rename columns for readability
        max_streaks.rename(
            columns={
                "consecutive_days": "Max_Simultaneous_Streak_Days",
                "start_date": "Streak_Start_Date",
                "end_date": "Streak_End_Date"
            },
            inplace=True
        )

        # Format dates as YYYY-MM-DD
        max_streaks["Streak_Start_Date"] = (
            max_streaks["Streak_Start_Date"]
            .dt.strftime("%Y-%m-%d")
        )

        max_streaks["Streak_End_Date"] = (
            max_streaks["Streak_End_Date"]
            .dt.strftime("%Y-%m-%d")
        )

        # Display the final result
        print("================================================================================")
        print(" MAXIMUM CONSECUTIVE DAYS (SIMULTANEOUS IN ACTIVITY, MOOD & SOCIAL)")
        print("================================================================================")
        print(max_streaks.to_string(index=False))
        print("\n")

Add Geo Information

In [ ]:
target_files = [
    "Initial_activity_file_New.csv",
    "Mood_file_New.csv",
    "Social_file_New.csv",
    "Geo_file.csv"
]

dfs_days = []
all_files_exist = True

# Load each file, extract unique recording days per participant,
# and store them in a list of DataFrames
for file_name in target_files:
    file_path = SAMSUNG_DIR / file_name

    if file_path.exists():
        df = pd.read_csv(file_path)

        # Extract the date only, ignoring the time
        df["date"] = pd.to_datetime(df["timestamp"]).dt.normalize()

        # Keep only unique recording days for each participant
        unique_days = df[["user_id", "date"]].drop_duplicates()
        dfs_days.append(unique_days)

    else:
        print(f"[ERROR] File not found: {file_path}")
        all_files_exist = False


if all_files_exist and len(dfs_days) == 4:

    # Keep only the days that are present in all four datasets
    intersected_days = dfs_days[0]

    for i in range(1, len(dfs_days)):
        intersected_days = pd.merge(
            intersected_days,
            dfs_days[i],
            on=["user_id", "date"],
            how="inner"
        )

    # Sort by participant and date
    intersected_days = intersected_days.sort_values(
        ["user_id", "date"]
    )

    if intersected_days.empty:
        print(
            "There are no days in which participants have data "
            "in all four files simultaneously."
        )

    else:
        # Calculate the difference in days between consecutive records
        intersected_days["day_diff"] = (
            intersected_days.groupby("user_id")["date"]
            .diff()
            .dt.days
        )

        # Identify streaks:
        # a new streak starts whenever the difference is not exactly one day
        intersected_days["streak_id"] = (
            intersected_days["day_diff"] != 1
        ).cumsum()

        # Calculate streak length, start date and end date
        streak_stats = (
            intersected_days.groupby(["user_id", "streak_id"])
            .agg(
                consecutive_days=("date", "size"),
                start_date=("date", "min"),
                end_date=("date", "max")
            )
            .reset_index()
        )

        # Find the longest simultaneous streak for each participant
        max_streak_idx = (
            streak_stats.groupby("user_id")["consecutive_days"]
            .idxmax()
        )

        max_streaks = (
            streak_stats.loc[max_streak_idx]
            .drop(columns=["streak_id"])
        )

        # Rename columns for readability
        max_streaks.rename(
            columns={
                "consecutive_days": "Max_Simultaneous_Streak_Days",
                "start_date": "Streak_Start_Date",
                "end_date": "Streak_End_Date"
            },
            inplace=True
        )

        # Format dates as YYYY-MM-DD
        max_streaks["Streak_Start_Date"] = (
            max_streaks["Streak_Start_Date"]
            .dt.strftime("%Y-%m-%d")
        )

        max_streaks["Streak_End_Date"] = (
            max_streaks["Streak_End_Date"]
            .dt.strftime("%Y-%m-%d")
        )

        # Display the final result
        print("================================================================================")
        print(" MAXIMUM CONSECUTIVE DAYS (SIMULTANEOUS IN ACTIVITY, MOOD, SOCIAL & GEO)")
        print("================================================================================")
        print(max_streaks.to_string(index=False))
        print("\n")

Compare with previous work

In [ ]:
target_files = [
    "Activity_file.csv",
    "Geo_file.csv",
    "Social_file.csv",
    "Mood_file.csv"
]

dfs_days = []
all_files_exist = True

# Load the recording days available in each file
for file_name in target_files:
    file_path = SAMSUNG_DIR / file_name

    if file_path.exists():
        df = pd.read_csv(file_path)

        # Extract the date from each timestamp
        df["date"] = pd.to_datetime(df["timestamp"]).dt.normalize()

        # Keep only unique recording days for each participant
        unique_days = df[["user_id", "date"]].drop_duplicates()
        dfs_days.append(unique_days)

    else:
        print(f"[ERROR] File not found: {file_path}")
        all_files_exist = False


if all_files_exist and len(dfs_days) == len(target_files):

    # Keep only the days that are present in all four datasets
    intersected_days = dfs_days[0]

    for i in range(1, len(dfs_days)):
        intersected_days = pd.merge(
            intersected_days,
            dfs_days[i],
            on=["user_id", "date"],
            how="inner"
        )

    # Sort by participant and date
    intersected_days = intersected_days.sort_values(
        ["user_id", "date"]
    )

    if intersected_days.empty:
        print(
            "There are no days in which participants have data "
            "in all four files simultaneously."
        )

    else:
        # Calculate the difference in days between consecutive records
        intersected_days["day_diff"] = (
            intersected_days.groupby("user_id")["date"]
            .diff()
            .dt.days
        )

        # Identify streaks:
        # a new streak starts whenever the difference is not exactly one day
        intersected_days["streak_id"] = (
            intersected_days["day_diff"] != 1
        ).cumsum()

        # Calculate streak length, start date and end date
        streak_stats = (
            intersected_days.groupby(["user_id", "streak_id"])
            .agg(
                consecutive_days=("date", "size"),
                start_date=("date", "min"),
                end_date=("date", "max")
            )
            .reset_index()
        )

        # Find the longest simultaneous streak for each participant
        max_streak_idx = (
            streak_stats.groupby("user_id")["consecutive_days"]
            .idxmax()
        )

        max_streaks = (
            streak_stats.loc[max_streak_idx]
            .drop(columns=["streak_id"])
        )

        # Rename columns for readability
        max_streaks.rename(
            columns={
                "consecutive_days": "Max_Simultaneous_Streak_Days",
                "start_date": "Streak_Start_Date",
                "end_date": "Streak_End_Date"
            },
            inplace=True
        )

        # Format dates as YYYY-MM-DD
        max_streaks["Streak_Start_Date"] = (
            max_streaks["Streak_Start_Date"]
            .dt.strftime("%Y-%m-%d")
        )

        max_streaks["Streak_End_Date"] = (
            max_streaks["Streak_End_Date"]
            .dt.strftime("%Y-%m-%d")
        )

        # Display the final result
        print("================================================================================")
        print(" MAXIMUM CONSECUTIVE DAYS - PREVIOUS DATA FILES (ALL 4 FILES)")
        print("================================================================================")
        print(max_streaks.to_string(index=False))
        print("\n")